# SSM <-> static MLP <-> ODE equivalence check

PhasorDense exposes three separate forward dispatches that *should*
agree on a constant input at steady state:

1. **Static MLP** (2D Phase): `out = complex_to_angle(W * exp(iπ·x) + bias)`.
   Single matrix-vector product in the complex domain. No SSM dynamics.
2. **Discrete SSM** (3D Phase): `out = causal_conv_dirac(x_repeated, W, λ, ω, T)`.
   Per-period Dirac kicks accumulated through the resonant kernel.
3. **Continuous ODE** (SpikingCall / CurrentCall): solve
   `dz/dt = k·z + W·I(t) + bias_current(t)` with `I(t)` a phase-encoded
   spike-train, integrated over many periods.

If we feed the same constant input to all three (for the SSM and ODE,
input is *repeated* across time), they should converge to the same
output as L (number of periods) grows. This notebook checks that —
both as a correctness test of the codebase and as a way to expose any
sign / phase-convention drift between dispatches.

# What this notebook produces

- A side-by-side scatter of phases from each path
- Convergence curves: ‖SSM_last − static‖ as a function of L
- The same vs the ODE
- Effect of decay rate (λ) and bias on the convergence
- Per-channel max error to surface anomalous channels

# What we expect

- Static and SSM should agree to within roundoff at large L *if* the
  three dispatches share the same phase convention for how a constant
  input phase maps to a complex value driving the dynamics.
- If they don't agree, the difference (e.g. a fixed π shift, a
  sign flip, an x-dependent magnitude factor) is informative — it
  identifies which encoding convention is "right" and lets us patch
  the others to match.

## 1. Setup

In [ ]:
using PhasorNetworks
using Lux, Random, Statistics
using CUDA, LuxCUDA
using Plots
using Random: Xoshiro
using Statistics: mean, std

# Use the same SpikingArgs the rest of the codebase uses (test/runtests.jl)
using DifferentialEquations: Tsit5
using SciMLSensitivity: BacksolveAdjoint, ZygoteVJP

const epsilon_test = 0.025

In [ ]:
# Knobs
const D            = 8           # small for visualization
const C_IN         = 8
const B            = 4           # batch of random inputs
const L_MAX        = 200         # max sequence length to test SSM convergence
const L_GRID       = [10, 25, 50, 100, 200]  # convergence-vs-L sweep
const LOG_NEG_LAMBDA = log(0.1)  # -> α = 0.1
const USE_BIAS     = true
const T            = 1.0f0       # period
USE_CUDA = false                  # CPU keeps the comparison reproducible
gdev = USE_CUDA ? gpu_device() : cpu_device()
@info "device" string(gdev)

## 2. Build a fixed PhasorDense layer

Same parameters used for all three dispatches. Layer is small (D=8,
C_in=8) for tractable inspection.

In [ ]:
rng = Xoshiro(0)
layer = PhasorDense(C_IN => D;
                    use_bias = USE_BIAS,
                    init_log_neg_lambda = LOG_NEG_LAMBDA)
ps_cpu, st_cpu = Lux.setup(rng, layer)
ps = ps_cpu |> gdev
st = st_cpu |> gdev

# Random input: B independent samples, each is a length-C_IN phase vector.
rng_in = Xoshiro(42)
x_2d = Phase.(2f0 .* rand(rng_in, Float32, C_IN, B) .- 1f0)        # (C_in, B)
println("Input shape: ", size(x_2d), " eltype: ", eltype(x_2d))
println("Input phases (first sample):")
println("  ", round.(Float32.(x_2d[:, 1]); digits = 3))

## 3. Path A — 2D Phase (static MLP)

In [ ]:
x_2d_dev = x_2d |> gdev
out_2d, _ = layer(x_2d_dev, ps, st)
out_2d_h = Array(out_2d)
println("out_2d size: ", size(out_2d_h), " eltype: ", eltype(out_2d_h))
println("out_2d (sample 1): ", round.(Float32.(out_2d_h[:, 1]); digits = 3))

## 4. Path B — 3D Phase SSM (constant input over L periods)

Tile the 2D input into `(C_in, L, B)` so the layer sees the same input
phase at every period. Take the last timestep as the "steady-state"
output.

In [ ]:
function tile_3d(x_2d::AbstractArray{<:Phase, 2}, L::Int)
    C, B = size(x_2d)
    x_3d = Array{Phase, 3}(undef, C, L, B)
    for t in 1:L
        x_3d[:, t, :] .= x_2d
    end
    return x_3d
end

# Run for max L; we'll slice intermediate timesteps for the convergence sweep.
x_3d_dev = tile_3d(x_2d, L_MAX) |> gdev
out_3d, _ = layer(x_3d_dev, ps, st)                                 # (D, L, B) Phase
out_3d_h = Array(out_3d)
out_3d_last = out_3d_h[:, end, :]                                    # (D, B) — "steady state"
println("out_3d size: ", size(out_3d_h))
println("out_3d_last (sample 1): ", round.(Float32.(out_3d_last[:, 1]); digits = 3))
println("vs out_2d   (sample 1): ", round.(Float32.(out_2d_h[:, 1]); digits = 3))

## 5. Path C — Continuous ODE via SpikingCall

Convert the constant input into a periodic spike train (one spike per
period at the time encoding each phase), build a CurrentCall, solve
the ODE for `repeats` periods, take the final period's output.

In [ ]:
# Spiking args matching test/runtests.jl
spk_args = SpikingArgs(t_window = 0.01,
                        threshold = 0.001,
                        solver = Tsit5(),
                        solver_args = Dict(:adaptive => false,
                                           :dt => 0.005,
                                           :sensealg => BacksolveAdjoint(autojacvec = ZygoteVJP()),
                                           :save_start => true))

# Build a SpikeTrain from the constant input phases, repeated for `n_repeats` periods.
n_repeats = 50
train = phase_to_train(x_2d; spk_args = spk_args, repeats = n_repeats)
tspan = (0.0f0, Float32(spk_args.t_period * n_repeats))
sc = SpikingCall(train, spk_args, tspan)

# CurrentCall dispatch returns (D, n_repeats, B) Phase via period-sampling
# when init_mode != :default; for :default mode it returns per-solver-step
# output. Layer's init_mode here is :default by default, so we'd get the
# per-step path. To get period-sampled output we'd need init_mode=:hippo
# OR we'd have to manually build the layer with init_mode that triggers
# SSM-mode sampling. For this experiment, easiest: call CurrentCall directly.
cc = CurrentCall(sc)
out_ode_raw, _ = layer(cc, ps, st)

# Inspect the type of out_ode_raw:
println("out_ode_raw type: ", typeof(out_ode_raw))
println("size: ", size(out_ode_raw))

## 6. Direct comparison: static vs SSM

For each (channel, batch sample), plot static phase vs SSM phase.
Perfect agreement = points lie on the line y=x.

In [ ]:
scatter(vec(Float32.(out_2d_h)), vec(Float32.(out_3d_last));
        xlabel = "static MLP phase (out_2d)",
        ylabel = "SSM steady-state phase (out_3d[end])",
        title = "Static vs SSM (D=$D, C_IN=$C_IN, B=$B, L=$L_MAX)",
        ms = 5, alpha = 0.7, label = "channels × samples",
        xlim = (-1.05, 1.05), ylim = (-1.05, 1.05))
plot!([-1, 1], [-1, 1]; ls = :dash, color = :gray, label = "y=x")

## 7. Convergence: how does SSM approach static as L grows?

For each L in `L_GRID`, compute `‖out_3d[:, L_idx, :] − out_2d‖` (mean
absolute error). If they're equivalent at steady state, the curve
should approach 0 as L grows.

In [ ]:
# Use the already-computed (D, L_MAX, B) tensor — slice at each L.
errors = Float32[]
for L in L_GRID
    err = mean(abs.(Float32.(out_3d_h[:, L, :]) .- Float32.(out_2d_h)))
    push!(errors, err)
end
println("L      mean |ΔPhase|")
for (l, e) in zip(L_GRID, errors)
    println("  $(rpad(l, 5))  $(round(e; digits = 5))")
end
plot(L_GRID, errors; marker = :circle, lw = 2,
     xlabel = "L (periods)", ylabel = "mean |out_3d - out_2d|",
     title = "SSM convergence to static MLP",
     yscale = :log10)

## 8. Effect of decay rate λ

Convergence speed should depend on |λ|: faster decay → quicker
saturation of the geometric sum → faster approach to steady state.
Build the same layer at three different decay rates and compare.

In [ ]:
λ_grid = [log(0.5), log(0.1), log(0.02)]    # α = 0.5, 0.1, 0.02 (fast/medium/slow)

results_by_λ = Dict()
for log_α in λ_grid
    α = exp(log_α)
    layer_λ = PhasorDense(C_IN => D; use_bias = USE_BIAS, init_log_neg_lambda = log_α)
    ps_λ = Lux.setup(Xoshiro(0), layer_λ)[1] |> gdev
    st_λ = Lux.setup(Xoshiro(0), layer_λ)[2] |> gdev
    out_2d_λ, _ = layer_λ(x_2d |> gdev, ps_λ, st_λ)
    out_3d_λ, _ = layer_λ(tile_3d(x_2d, L_MAX) |> gdev, ps_λ, st_λ)
    out_2d_h_λ = Array(out_2d_λ)
    out_3d_h_λ = Array(out_3d_λ)
    errs_λ = [mean(abs.(Float32.(out_3d_h_λ[:, L, :]) .- Float32.(out_2d_h_λ))) for L in L_GRID]
    results_by_λ[α] = errs_λ
    println("α=$(round(α; digits=3)): errors over L_GRID = ", round.(errs_λ; digits = 4))
end

plt = plot(xlabel = "L (periods)", ylabel = "mean |Δphase|",
           title = "Convergence by decay rate", yscale = :log10)
for α in sort(collect(keys(results_by_λ)); rev = true)
    plot!(plt, L_GRID, results_by_λ[α]; marker = :circle, lw = 2, label = "α = $(round(α; digits=3))")
end
plt

## 9. Effect of bias

The bias-per-period fix should let the SSM and static MLP agree
better, because the bias contribution scales with the kernel sum
(matching the static MLP's "bias added once" semantics in steady
state). Compare with vs without bias.

In [ ]:
function final_error(use_bias::Bool, log_α::Real, L::Int = L_MAX)
    layer_b = PhasorDense(C_IN => D; use_bias = use_bias, init_log_neg_lambda = log_α)
    ps_b = Lux.setup(Xoshiro(0), layer_b)[1] |> gdev
    st_b = Lux.setup(Xoshiro(0), layer_b)[2] |> gdev
    out_2d_b = Array(layer_b(x_2d |> gdev, ps_b, st_b)[1])
    out_3d_b = Array(layer_b(tile_3d(x_2d, L) |> gdev, ps_b, st_b)[1])
    return mean(abs.(Float32.(out_3d_b[:, end, :]) .- Float32.(out_2d_b)))
end

println("Bias    α=0.5   α=0.1   α=0.02")
for use_bias in (false, true)
    errs = [round(final_error(use_bias, log_α); digits = 4) for log_α in λ_grid]
    println("  $(rpad(string(use_bias), 6)) $(rpad(errs[1], 7)) $(rpad(errs[2], 7)) $(errs[3])")
end

## 10. Per-channel error breakdown

Some channels may converge faster than others (different |W[c,:]|, sign
of weight sum, etc.). Histogram per-channel max error to see if all are
behaving the same or if a few outliers dominate.

In [ ]:
per_chan_err = vec(maximum(abs.(Float32.(out_3d_last) .- Float32.(out_2d_h)); dims = 2))
println("Per-channel max |Δphase|: ", round.(per_chan_err; digits = 4))

bar(per_chan_err;
    xlabel = "channel", ylabel = "max |Δphase| across batch",
    title = "Per-channel SSM-vs-static error (L=$L_MAX, α=$(round(exp(LOG_NEG_LAMBDA); digits = 3)))",
    legend = false)

## 11. Side-by-side: per-sample, per-channel phase comparison

In [ ]:
# 4-row plot: one per batch sample
plots_b = []
for b in 1:B
    p = plot(1:D, Float32.(out_2d_h[:, b]); marker = :circle, lw = 0,
             ms = 6, label = "static (out_2d)",
             xlabel = "channel", ylabel = "phase",
             title = "sample $b", ylim = (-1.05, 1.05))
    plot!(p, 1:D, Float32.(out_3d_last[:, b]); marker = :diamond, lw = 0,
          ms = 6, label = "SSM (out_3d[end])")
    push!(plots_b, p)
end
plot(plots_b...; layout = (B, 1), size = (700, 200 * B))

## 12. ODE comparison (best-effort)

The CurrentCall dispatch returns different shapes depending on
`init_mode`. For `:default` mode (which we used), it returns
per-solver-step output. To get a clean comparison with the SSM
sample-at-period-boundaries output, we either need to slice the ODE
solution at period times manually, or use `init_mode=:hippo` which
triggers period-sampling. We'll do both, but mainly use the
period-sampled form for parity.

In [ ]:
# Build a layer that uses the SSM-mode sampling in the ODE path.
layer_h = PhasorDense(C_IN => D; use_bias = USE_BIAS,
                       init_log_neg_lambda = LOG_NEG_LAMBDA, init_mode = :hippo)
ps_h_cpu, st_h_cpu = Lux.setup(Xoshiro(0), layer_h)
ps_h = ps_h_cpu |> gdev
st_h = st_h_cpu |> gdev
x_2d_h_input = x_2d |> gdev
out_2d_h_layer = Array(layer_h(x_2d_h_input, ps_h, st_h)[1])

# Re-use the SpikingCall built above, but for layer_h:
out_ode_h, _ = layer_h(sc, ps_h, st_h)
println("out_ode_h type: ", typeof(out_ode_h), " size: ", size(out_ode_h))

# Compare last-period ODE output to static
if eltype(out_ode_h) <: Phase && ndims(out_ode_h) == 3
    out_ode_last = Array(out_ode_h[:, end, :])
    err_ode = mean(abs.(Float32.(out_ode_last) .- Float32.(out_2d_h_layer)))
    println("ODE-last vs static error: ", round(err_ode; digits = 4))
    println("Sample 1:")
    println("  static:    ", round.(Float32.(out_2d_h_layer[:, 1]); digits = 3))
    println("  ODE last:  ", round.(Float32.(out_ode_last[:, 1]); digits = 3))
else
    println("ODE returned per-solver-step output; skipping period-sample comparison.")
    println("(Use init_mode=:hippo for the period-sampled ODE path.)")
end

## Notes on what to look for

If sections 7–9 show errors plateauing well above 0 (instead of
decaying toward roundoff), the three dispatches have a **structural
mismatch** — most likely an x-dependent encoding (`exp(k·dt)` in
SSM vs `exp(iπ·θ)` in static) that doesn't reduce to the same value
at steady state regardless of L.

If errors decay smoothly to ~0 as L grows, the dispatches agree at
steady state — and the bias-per-period fix is doing its job.

If only **some channels** have large error in section 10, the issue
is likely numerical (those channels have small |z|, where phase is
ill-defined) rather than structural.

## Suggested follow-ups

- If we see a fixed π shift between static and SSM, look at the
  Dirac encoding's phase convention — e.g., maybe the spike should
  arrive at `t_s = θ·T` rather than `(θ+1)/2·T`.
- If we see a systematic magnitude-dependent phase shift, the
  `exp(λ·dt)` magnitude factor in the Dirac encoding is the culprit.
- If errors decay slowly with L for slow decay (small α), it's just
  the geometric sum's slow saturation — fine in principle, but
  practically constrains how long the SSM needs to run to "converge".